## 07 - Hedge Detection

This notebook measures hedging language in each earnings call transcript.

### What is hedging?
Hedging is cautious language CEOs use to protect themselves legally
while still sounding confident publicly. Words like "may", "could",
"uncertain" signal that the CEO is not fully confident in their outlook.

### What we measure?
- Hedge count: total hedge words in transcript
- Hedge ratio: hedge words as percentage of total words
- Top hedge words: which specific words appear most often

### Why this matters?
High sentiment + high hedging = the "lying signal"
The CEO sounds positive but their word choices reveal uncertainty.

In [1]:
# Import libraries
import sqlite3
import pandas as pd
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

# Connect to database
conn = sqlite3.connect("../data/earnings_research.db")

# Load all transcripts
transcripts = pd.read_sql("""
    SELECT id, ticker, quarter, date, period, raw_text
    FROM transcripts
    ORDER BY date
""", conn)

print(f"Loaded {len(transcripts)} transcripts")


Loaded 14 transcripts


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\KOMAL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\KOMAL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
# Define our hedge word dictionary 
# These are words that signal uncertainty or caution in financial language

hedge_words = [
    # Uncertainty word
    "may", "might", "could", "would", "should",
    "possibly", "perhaps", "potentially", "likely", "unlikely",

    # Cautious verbs
    "expect", "believe", "anticipate", "estimate", "assume",
    "suggest", "appear", "seem", "tend",
    
    # Uncertainty nouns
    "uncertainty", "risk", "challenge", "concern", "volatility",
    "headwind", "pressure", "difficulty", "weakness", "slowdown",
    
    # Hedging phrases (single words from them)
    "subject", "contingent", "dependent", "conditional",
    "approximately", "roughly", "around", "about",
    
    # Forward looking disclaimers
    "forward", "outlook", "guidance", "forecast", "projection"
]

print(f"Total hedge words in dictionary: {len(hedge_words)}")
print(hedge_words)

Total hedge words in dictionary: 42
['may', 'might', 'could', 'would', 'should', 'possibly', 'perhaps', 'potentially', 'likely', 'unlikely', 'expect', 'believe', 'anticipate', 'estimate', 'assume', 'suggest', 'appear', 'seem', 'tend', 'uncertainty', 'risk', 'challenge', 'concern', 'volatility', 'headwind', 'pressure', 'difficulty', 'weakness', 'slowdown', 'subject', 'contingent', 'dependent', 'conditional', 'approximately', 'roughly', 'around', 'about', 'forward', 'outlook', 'guidance', 'forecast', 'projection']


In [5]:
# Build hedge detection function
from collections import Counter

def detect_hedges(text, hedge_words):
    """
    Takes a transcript text and returns hedge metrics.
    """
    # Tokenize into individual words and lowercase
    words = word_tokenize(text.lower())

    # Total word count
    total_words = len(words)

    # Find hedges words
    found_hedges = [word for word in words if word in hedge_words]

    # Count each hedge word
    hedge_counts = Counter(found_hedges)

    # Calculate metrics
    hedge_count = len(found_hedges)
    hedge_ratio = round(hedge_count / total_words, 4) if total_words > 0 else 0
    top_5_hedges = hedge_counts.most_common(5)

    return {
         "total_words": total_words,
         "hedge_count": hedge_count,
         "hedge_ratio": hedge_ratio,
         "top_hedges": top_5_hedges
    }

# Test on Apple Q1 2020
test_text = transcripts[transcripts["ticker"] == "AAPL"].iloc[0]["raw_text"]
result = detect_hedges(test_text, hedge_words)
print(f"Total words: {result['total_words']}")
print(f"Hedge count: {result['hedge_count']}")
print(f"Hedge ratio: {result['hedge_ratio']}")
print(f"Top 5 hedge words: {result['top_hedges']}")
    

Total words: 3609
Hedge count: 27
Hedge ratio: 0.0075
Top 5 hedge words: [('expect', 6), ('about', 4), ('around', 4), ('outlook', 2), ('guidance', 2)]


In [6]:
# Run hedge detection on all 14 transcripts
hedge_results = []

for _, row in transcripts.iterrows():
    result = detect_hedges(row["raw_text"], hedge_words)
    
    hedge_results.append({
        "ticker": row["ticker"],
        "quarter": row["quarter"],
        "date": row["date"],
        "period": row["period"],
        "total_words": result["total_words"],
        "hedge_count": result["hedge_count"],
        "hedge_ratio": result["hedge_ratio"],
        "top_hedges": str(result["top_hedges"])
    })
    
    print(f"✓ {row['ticker']} {row['quarter']} → hedge ratio: {result['hedge_ratio']} ({result['hedge_count']} hedges in {result['total_words']} words)")

print("\nAll transcripts analyzed!")

✓ AAPL Q1_2020 → hedge ratio: 0.0075 (27 hedges in 3609 words)
✓ JNJ Q1_2020 → hedge ratio: 0.0136 (155 hedges in 11386 words)
✓ JPM Q1_2020 → hedge ratio: 0.0139 (62 hedges in 4454 words)
✓ MSFT Q1_2020 → hedge ratio: 0.0091 (42 hedges in 4627 words)
✓ AAPL Q2_2020 → hedge ratio: 0.0105 (48 hedges in 4574 words)
✓ AMZN Q1_2020 → hedge ratio: 0.0127 (19 hedges in 1497 words)
✓ XOM Q1_2020 → hedge ratio: 0.0118 (55 hedges in 4672 words)
✓ JPM Q2_2020 → hedge ratio: 0.0146 (77 hedges in 5260 words)
✓ MSFT Q4_2022 → hedge ratio: 0.0166 (183 hedges in 11049 words)
✓ AAPL Q4_2022 → hedge ratio: 0.0141 (140 hedges in 9917 words)
✓ MSFT Q2_2023 → hedge ratio: 0.0162 (180 hedges in 11144 words)
✓ NVDA Q2_2023 → hedge ratio: 0.0154 (157 hedges in 10174 words)
✓ AAPL Q1_2025 → hedge ratio: 0.0124 (127 hedges in 10215 words)
✓ AMZN Q1_2025 → hedge ratio: 0.011 (49 hedges in 4440 words)

All transcripts analyzed!


In [9]:
# Combine sentiment scores with hedge ratios
df_hedges = pd.DataFrame(hedge_results)

# Load sentiment scores from database
df_sentiment = pd.read_sql("""
    SELECT ticker, quarter, avg_compound
    FROM sentiment_scores
""", conn)

# Merge the two datasets
df_combined = pd.merge(df_hedges, df_sentiment, on=["ticker", "quarter"])

# Calculate the "lying signal" - high sentiment + high hedging
df_combined["sentiment_hedge_gap"] = round(
    df_combined["avg_compound"] - (df_combined["hedge_ratio"] * 10), 4
)

# Sort by most interesting cases
df_combined = df_combined.sort_values("avg_compound", ascending=False)

print(df_combined[["ticker", "quarter", "period", "avg_compound", "hedge_ratio", "sentiment_hedge_gap"]].to_string(index=False))

ticker quarter         period  avg_compound  hedge_ratio  sentiment_hedge_gap
  AAPL Q2_2020    COVID Crash        0.3193       0.0105               0.2143
  MSFT Q1_2020      Pre-COVID        0.3113       0.0091               0.2203
  AAPL Q1_2020      Pre-COVID        0.2968       0.0075               0.2218
   JNJ Q1_2020      Pre-COVID        0.2844       0.0136               0.1484
  AMZN Q1_2025   Tariff Shock        0.2780       0.0110               0.1680
  AAPL Q1_2025   Tariff Shock        0.2703       0.0124               0.1463
  MSFT Q4_2022 Rate Hike Peak        0.2529       0.0166               0.0869
  MSFT Q2_2023        AI Boom        0.2492       0.0162               0.0872
  AAPL Q4_2022 Rate Hike Peak        0.2228       0.0141               0.0818
   XOM Q1_2020      Pre-COVID        0.1838       0.0118               0.0658
   JPM Q1_2020      Pre-COVID        0.1794       0.0139               0.0404
  NVDA Q2_2023        AI Boom        0.1511       0.0154        

In [10]:
# Save combined results to database
df_combined.to_sql("hedge_analysis", conn, if_exists="replace", index=False)
conn.commit()

print("Hedge analysis saved to database!")

# Close connection
conn.close()
print("Database connection closed.")

Hedge analysis saved to database!
Database connection closed.
